# EEG · Causal Conv1D — Multi-Trial Training & Evaluation

Trains on **multiple trials** and evaluates with **leave-one-trial-out (LOTO) CV**.

## Why trial-grouped CV matters

Consecutive windows share T-1 samples — random splits leak adjacent windows
across folds and inflate scores.  Grouping by trial forces the model to
generalise *across* recording sessions, which is what matters in live use.

```
Trials 0–7  ───►  Leave-One-Trial-Out CV (Optuna inner + LOTO outer)
Trials 8–9  ───►  Held-out test set  (never seen during training or tuning)
```

| Step | Trials | Purpose |
|---|---|---|
| Optuna inner CV | 0–7, GroupKFold(8) | Hyperparameter search |
| LOTO outer CV | 0–7, LeaveOneGroupOut | Unbiased AUC + optimal threshold |
| **Test** | **8, 9** | **Final held-out evaluation** |

**Normalisation:** per-window z-score per channel — stateless, no global
scaler needed, robust to session-level amplitude drift.

In [1]:
import warnings

warnings.filterwarnings("ignore")
import os

import numpy as np
import optuna
import pandas as pd
import torch
import torch.nn as nn
from mne.filter import filter_data
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GroupKFold, LeaveOneGroupOut
from torch.utils.data import DataLoader, TensorDataset

optuna.logging.set_verbosity(optuna.logging.WARNING)

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

DARK = "#0a0e17"
CARD = "#111827"
EDGE = "#1f2937"
TEAL = "#00e5cc"
CORAL = "#ff4f5e"
GOLD = "#ffc947"
LIME = "#a8ff3e"
WHITE = "#f0f4ff"
GRAY = "#6b7280"

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"Device : {DEVICE}")
print("All imports OK.")

Device : mps
All imports OK.


## 1 · Configuration

In [2]:
FS = 250
SUBJECT = 0
TACHE = "spt"
DATA_PATH = f"../data/subject{SUBJECT}/"

TRAIN_TRIALS = list(range(8))  # 0–7  →  LOTO CV
TEST_TRIALS = [8, 9]  # truly held out

N_TRIALS_OPTUNA = 20  # Optuna budget (conv1d is slower than RF)
APPLY_CLEANING = False

print(f"Train trials : {TRAIN_TRIALS}")
print(f"Test  trials : {TEST_TRIALS}")
print(f"Device       : {DEVICE}")

Train trials : [0, 1, 2, 3, 4, 5, 6, 7]
Test  trials : [8, 9]
Device       : mps


## 2 · Load & Preprocess All Trials

Each trial is band-pass filtered (0.1–60 Hz) individually,
then concatenated with a `trial_id` group vector.

In [3]:
def load_trial(subject, tache, trial, data_path, eeg_cols=None):
    df = pd.read_csv(f"{data_path}/subject_{subject}_tache_{tache}_trial_{trial}.csv")
    cols = [c for c in df.columns if c not in ("button", "timestamp")]
    if eeg_cols is not None:
        assert cols == eeg_cols, f"Trial {trial} channel mismatch"
    df.iloc[:, 1:-1] = filter_data(df.iloc[:, 1:-1].values.T, FS, 0.1, 60).T
    return df, cols


def build_windows(
    X: np.ndarray,
    y: np.ndarray,
    groups: np.ndarray,
    T: int,
    per_window_norm: bool = True,
):
    """
    Causal sliding windows of length T for Conv1D.

    Parameters
    ----------
    X      : (N, C) raw EEG
    y      : (N,)   labels
    groups : (N,)   trial_id per sample
    T      : look-back in samples
    per_window_norm : z-score each window per channel independently

    Returns
    -------
    X_win  : (M, C, T)   Conv1D format — (batch, channels, time)
    y_win  : (M,)
    g_win  : (M,)        trial group for each window
    """
    N, C = X.shape
    wins, labels, grps = [], [], []
    for i in range(T, N):
        if groups[i] != groups[i - T]:  # window spans two trials — skip
            continue
        w = X[i - T : i].copy()  # (T, C)
        if per_window_norm:
            mu = w.mean(axis=0, keepdims=True)
            sd = w.std(axis=0, keepdims=True) + 1e-8
            w = (w - mu) / sd
        wins.append(w.T)  # (C, T) for Conv1D
        labels.append(y[i])
        grps.append(groups[i])
    return (
        np.array(wins, dtype=np.float32),
        np.array(labels, dtype=np.int64),
        np.array(grps, dtype=np.int64),
    )


print("Helper functions defined.")

Helper functions defined.


In [4]:
print("Loading train trials …")
EEG_COLS = None
X_parts, y_parts, g_parts = [], [], []

for t in TRAIN_TRIALS:
    df, cols = load_trial(SUBJECT, TACHE, t, DATA_PATH, eeg_cols=EEG_COLS)
    if EEG_COLS is None:
        EEG_COLS = cols
    X_parts.append(df[EEG_COLS].values.astype(np.float32))
    y_parts.append(df["button"].values.astype(int))
    g_parts.append(np.full(len(df), t, dtype=np.int64))
    bal = df["button"].value_counts().to_dict()
    print(f"  trial {t}: {len(df):>6,} samples  label={bal}")

X_train_raw = np.concatenate(X_parts, axis=0)
y_train_raw = np.concatenate(y_parts, axis=0)
g_train_raw = np.concatenate(g_parts, axis=0)

N_CHANNELS = len(EEG_COLS)
print(f"\nCombined train : {X_train_raw.shape}")
print(f"Channels       : {N_CHANNELS}  {EEG_COLS}")
print(f"Class balance  : {dict(zip(*np.unique(y_train_raw, return_counts=True)))}")

Loading train trials …
Setting up band-pass filter from 0.1 - 60 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.10
- Lower transition bandwidth: 0.10 Hz (-6 dB cutoff frequency: 0.05 Hz)
- Upper passband edge: 60.00 Hz
- Upper transition bandwidth: 15.00 Hz (-6 dB cutoff frequency: 67.50 Hz)
- Filter length: 8251 samples (33.004 s)

  trial 0: 14,244 samples  label={0: 10351, 1: 3893}
Setting up band-pass filter from 0.1 - 60 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.10
- Lower transition bandwidth: 0.10 Hz (-6 dB cutoff frequency: 0.05 Hz)
- Upper passband edge: 60.00

In [5]:
print("Loading test trials …")
X_test_parts, y_test_parts, g_test_parts = [], [], []

for t in TEST_TRIALS:
    df_t, _ = load_trial(SUBJECT, TACHE, t, DATA_PATH, eeg_cols=EEG_COLS)
    X_test_parts.append(df_t[EEG_COLS].values.astype(np.float32))
    y_test_parts.append(df_t["button"].values.astype(int))
    g_test_parts.append(np.full(len(df_t), t, dtype=np.int64))
    bal = df_t["button"].value_counts().to_dict()
    print(f"  trial {t}: {len(df_t):>6,} samples  label={bal}")

X_test_raw = np.concatenate(X_test_parts, axis=0)
y_test_raw = np.concatenate(y_test_parts, axis=0)
g_test_raw = np.concatenate(g_test_parts, axis=0)
print(f"\nCombined test  : {X_test_raw.shape}")

Loading test trials …
Setting up band-pass filter from 0.1 - 60 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.10
- Lower transition bandwidth: 0.10 Hz (-6 dB cutoff frequency: 0.05 Hz)
- Upper passband edge: 60.00 Hz
- Upper transition bandwidth: 15.00 Hz (-6 dB cutoff frequency: 67.50 Hz)
- Filter length: 8251 samples (33.004 s)

  trial 8: 17,884 samples  label={0: 15599, 1: 2285}
Setting up band-pass filter from 0.1 - 60 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.10
- Lower transition bandwidth: 0.10 Hz (-6 dB cutoff frequency: 0.05 Hz)
- Upper passband edge: 60.00 

## 3 · Causal Conv1D Architecture

```
Input  (batch, C, T)
  │
  ├─ Conv1d + causal padding + BN + act + Dropout   × n_layers
  │
  ├─ AdaptiveAvgPool1d(1)   →  (batch, last_filters)
  │
  └─ Linear(last_filters, 1)  →  logit  →  σ  →  P(open at T+1)
```

**Causal padding** (`pad = kernel_size - 1` on the left, trimmed on the right)
ensures each timestep only sees current and past inputs.

In [6]:
class _Trim(nn.Module):
    """Remove n timesteps from the right to enforce causal padding."""

    def __init__(self, n):
        super().__init__()
        self.n = n

    def forward(self, x):
        return x[..., : -self.n] if self.n > 0 else x


class CausalConv1D(nn.Module):
    ACTIVATIONS = {
        "relu": nn.ReLU,
        "leaky_relu": nn.LeakyReLU,
        "elu": nn.ELU,
        "gelu": nn.GELU,
    }

    def __init__(
        self,
        in_channels,
        n_filters,
        n_layers,
        kernel_size,
        grow_filters,
        dropout_rate,
        activation,
        use_bn,
    ):
        super().__init__()
        Act = self.ACTIVATIONS[activation]
        layers, in_ch = [], in_channels

        for i in range(n_layers):
            out_ch = min(n_filters * (2**i) if grow_filters else n_filters, 256)
            padding = kernel_size - 1
            layers += [
                nn.Conv1d(in_ch, out_ch, kernel_size=kernel_size, padding=padding),
                _Trim(padding),
            ]
            if use_bn:
                layers.append(nn.BatchNorm1d(out_ch))
            layers.append(Act())
            if dropout_rate > 0:
                layers.append(nn.Dropout(dropout_rate))
            in_ch = out_ch

        self.conv_stack = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Linear(in_ch, 1)

    def forward(self, x):  # x: (B, C, T)
        x = self.conv_stack(x)  # (B, last_ch, T)
        return self.head(self.pool(x).squeeze(-1)).squeeze(1)  # (B,)


# Sanity check
_m = CausalConv1D(16, 32, 3, 5, True, 0.3, "relu", True)
_x = torch.randn(8, 16, 64)
print(f"Input : {_x.shape}  →  Output : {_m(_x).shape}")
print(f"Params: {sum(p.numel() for p in _m.parameters()):,}")

Input : torch.Size([8, 16, 64])  →  Output : torch.Size([8])
Params: 54,561


## 4 · Training Helpers

In [7]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE).float()
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()


@torch.no_grad()
def predict_proba(model, X_t, batch_size=512):
    model.eval()
    loader = DataLoader(
        TensorDataset(X_t, torch.zeros(len(X_t))), batch_size=batch_size
    )
    return torch.cat(
        [torch.sigmoid(model(xb.to(DEVICE))).cpu() for xb, _ in loader]
    ).numpy()


def build_optimizer(model, name, lr, wd):
    if name == "adamw":
        return torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    if name == "sgd":
        return torch.optim.SGD(model.parameters(), lr=lr, weight_decay=wd, momentum=0.9)
    return torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)


def run_fold(
    X_tr_t,
    y_tr_t,
    X_va_t,
    y_va_t,
    n_filters,
    n_layers,
    kernel_size,
    grow_filters,
    dropout,
    activation,
    use_bn,
    opt_name,
    lr,
    wd,
    batch_size,
    max_epochs=50,
    patience=8,
):
    """Train one CV fold; returns (best_auc, probs, history)."""
    pw = (
        torch.tensor([(y_tr_t == 0).sum() / max((y_tr_t == 1).sum(), 1)])
        .float()
        .to(DEVICE)
    )
    criterion = nn.BCEWithLogitsLoss(pos_weight=pw)

    model = CausalConv1D(
        X_tr_t.shape[1],
        n_filters,
        n_layers,
        kernel_size,
        grow_filters,
        dropout,
        activation,
        use_bn,
    ).to(DEVICE)
    optimizer = build_optimizer(model, opt_name, lr, wd)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs)
    loader = DataLoader(
        TensorDataset(X_tr_t, y_tr_t.float()),
        batch_size=batch_size,
        shuffle=True,
    )

    best_auc, best_state, pat_cnt, history = 0.0, None, 0, []
    for _ in range(max_epochs):
        train_epoch(model, loader, optimizer, criterion)
        scheduler.step()
        probs = predict_proba(model, X_va_t, batch_size)
        auc = float(roc_auc_score(y_va_t.numpy(), probs))
        history.append(auc)
        if auc > best_auc:
            best_auc = auc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            pat_cnt = 0
        else:
            pat_cnt += 1
        if pat_cnt >= patience:
            break

    model.load_state_dict(best_state)
    probs = predict_proba(model, X_va_t, batch_size)
    return best_auc, probs, history, model


print("Training helpers defined.")

Training helpers defined.


## 5 · Optuna Hyperparameter Search — GroupKFold by Trial

Each Optuna fold holds out one complete trial so no within-trial leakage
occurs during tuning.  `GroupKFold(n_splits=8)` → 8 folds, one per train trial.

| Hyperparameter | Range |
|---|---|
| **T** | 16 – 256 samples |
| `n_layers` | 1 – 5 |
| `n_filters` | 8 – 128 (log) |
| `kernel_size` | 3, 5, 7, 11 |
| `grow_filters` | True / False |
| `dropout` | 0.0 – 0.5 |
| `activation` | relu, elu, gelu |
| `use_bn` | True / False |
| `optimizer` | adam, adamw |
| `lr` | 1e-4 – 1e-2 (log) |
| `weight_decay` | 1e-5 – 1e-2 (log) |
| `batch_size` | 64, 128, 256 |

In [ ]:
cv_inner = GroupKFold(n_splits=len(TRAIN_TRIALS))


def objective(trial):
    T = trial.suggest_int("T", 16, 256, log=True)
    n_layers = trial.suggest_int("n_layers", 1, 5)
    n_filters = trial.suggest_int("n_filters", 8, 128, log=True)
    kernel_size = trial.suggest_categorical("kernel_size", [3, 5, 7, 11])
    grow_filters = trial.suggest_categorical("grow_filters", [True, False])
    dropout = trial.suggest_float("dropout", 0.0, 0.5)
    activation = trial.suggest_categorical("activation", ["relu", "elu", "gelu"])
    use_bn = trial.suggest_categorical("use_bn", [True, False])
    opt_name = trial.suggest_categorical("optimizer", ["adam", "adamw"])
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    wd = trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [64, 128, 256])

    X_win, y_win, g_win = build_windows(X_train_raw, y_train_raw, g_train_raw, T)
    X_t = torch.tensor(X_win)
    y_t = torch.tensor(y_win)

    fold_aucs = []
    for fold, (tr_idx, va_idx) in enumerate(cv_inner.split(X_win, y_win, groups=g_win)):
        auc, _, _, _ = run_fold(
            X_t[tr_idx],
            y_t[tr_idx],
            X_t[va_idx],
            y_t[va_idx],
            n_filters,
            n_layers,
            kernel_size,
            grow_filters,
            dropout,
            activation,
            use_bn,
            opt_name,
            lr,
            wd,
            batch_size,
        )
        fold_aucs.append(auc)
        trial.report(np.mean(fold_aucs), step=fold)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return float(np.mean(fold_aucs))


study = optuna.create_study(
    direction="maximize",
    sampler=TPESampler(seed=42),
    pruner=MedianPruner(n_startup_trials=8, n_warmup_steps=0),
)
study.optimize(objective, n_trials=N_TRIALS_OPTUNA, show_progress_bar=True)

print(f"\nBest trial : #{study.best_trial.number}")
print(f"Best AUC   : {study.best_value:.4f}")
print("\nBest params:")
for k, v in study.best_params.items():
    print(f"  {k:22s}: {v}")

Best trial: 2. Best value: 0.615109:  20%|█████████████████████▌                                                                                      | 4/20 [22:36<1:25:07, 319.22s/it]

## 6 · Extract Best Config & Build Full Train Windows

In [ ]:
p = study.best_params
T = p["T"]
n_layers = p["n_layers"]
n_filters = p["n_filters"]
kernel_size = p["kernel_size"]
grow_filters = p["grow_filters"]
dropout = p["dropout"]
activation = p["activation"]
use_bn = p["use_bn"]
opt_name = p["optimizer"]
lr = p["lr"]
wd = p["weight_decay"]
batch_size = p["batch_size"]

X_win, y_win, g_win = build_windows(X_train_raw, y_train_raw, g_train_raw, T)
X_torch = torch.tensor(X_win)
y_torch = torch.tensor(y_win)

print(f"Best T       : {T} samples  ({T/FS*1000:.0f} ms look-back)")
print(f"Dataset      : {X_win.shape}  →  {y_win.shape}")
print(f"Trial groups : {np.unique(g_win)}")
print(f"Conv layers  : {n_layers} × {n_filters} filters  kernel={kernel_size}")
print(f"Grow filters : {grow_filters}  BN: {use_bn}")
print(f"Activation   : {activation}  Dropout: {dropout:.3f}")
print(f"Optimizer    : {opt_name}  lr={lr:.2e}  wd={wd:.2e}  batch={batch_size}")

## 7 · Leave-One-Trial-Out (LOTO) Outer CV

Each fold trains on 7 trials and validates on 1 complete trial.  
Probabilities from all folds are concatenated to compute the optimal
decision threshold (Youden's J).

In [ ]:
logo = LeaveOneGroupOut()

loto_aucs = []
all_probs = np.zeros(len(y_win), dtype=np.float32)
all_preds = np.full(len(y_win), -1, dtype=int)
train_hists = []
best_models = {}  # held_trial → model state_dict

for held_trial, (tr_idx, va_idx) in zip(
    np.unique(g_win), logo.split(X_win, y_win, groups=g_win)
):

    auc, probs, hist, model = run_fold(
        X_torch[tr_idx],
        y_torch[tr_idx],
        X_torch[va_idx],
        y_torch[va_idx],
        n_filters,
        n_layers,
        kernel_size,
        grow_filters,
        dropout,
        activation,
        use_bn,
        opt_name,
        lr,
        wd,
        batch_size,
        max_epochs=60,
        patience=10,
    )
    all_probs[va_idx] = probs
    all_preds[va_idx] = (probs >= 0.5).astype(int)
    loto_aucs.append(auc)
    train_hists.append(hist)
    best_models[int(held_trial)] = {k: v.clone() for k, v in model.state_dict().items()}
    print(
        f"  held-out trial {int(held_trial):2d}  AUC={auc:.4f}  "
        f"epochs={len(hist)}  n_val={len(va_idx):,}"
    )

mask = all_preds >= 0
print(f"\nLOTO mean AUC : {np.mean(loto_aucs):.4f} ± {np.std(loto_aucs):.4f}")
print(f"Accuracy @ 0.5: {(all_preds[mask] == y_win[mask]).mean():.4f}")
print()
print(
    classification_report(y_win[mask], all_preds[mask], target_names=["Closed", "Open"])
)

## 8 · Optimal Threshold (Youden's J)

In [ ]:
fpr_cv, tpr_cv, thresholds_cv = roc_curve(y_win[mask], all_probs[mask])
best_thresh = float(thresholds_cv[np.argmax(tpr_cv - fpr_cv)])
preds_opt = (all_probs[mask] >= best_thresh).astype(int)
acc_opt = (preds_opt == y_win[mask]).mean()

print(f"Optimal threshold (Youden's J) : {best_thresh:.3f}")
print(f"Accuracy @ optimal threshold   : {acc_opt:.4f}")
print()
print(classification_report(y_win[mask], preds_opt, target_names=["Closed", "Open"]))

## 9 · Retrain on All Train Trials

A single final model is trained on all 8 train trials combined.  
This is the model saved to disk and used by the server.

In [ ]:
print("Retraining on all train trials …")
_, _, final_hist, final_model = run_fold(
    X_torch,
    y_torch,
    X_torch,
    y_torch,  # val = train for the final fit (no early stopping bias)
    n_filters,
    n_layers,
    kernel_size,
    grow_filters,
    dropout,
    activation,
    use_bn,
    opt_name,
    lr,
    wd,
    batch_size,
    max_epochs=60,
    patience=60,  # run full 60 epochs, no early stop
)
final_model.eval()
print(f"Final model trained for {len(final_hist)} epochs.")

## 10 · Held-Out Test Evaluation (Trials 8 & 9)

These trials were **never seen** during tuning, CV, or threshold selection.

In [ ]:
X_win_test, y_win_test, g_win_test = build_windows(
    X_test_raw, y_test_raw, g_test_raw, T
)
X_torch_test = torch.tensor(X_win_test)

probs_test = predict_proba(final_model, X_torch_test)
preds_test = (probs_test >= best_thresh).astype(int)
auc_test = roc_auc_score(y_win_test, probs_test)
acc_test = (preds_test == y_win_test).mean()

print("=" * 60)
print("  Held-out Test Results (trials 8 & 9)")
print("=" * 60)
print(f"  Windows : {X_win_test.shape[0]:,}")
print(f"  AUC     : {auc_test:.4f}")
print(f"  Accuracy: {acc_test:.4f}  (threshold={best_thresh:.3f})")
print()
print(classification_report(y_win_test, preds_test, target_names=["Closed", "Open"]))
print("=" * 60)

print("\nPer-trial breakdown:")
for t in TEST_TRIALS:
    m = g_win_test == t
    print(
        f"  trial {t}: AUC={roc_auc_score(y_win_test[m], probs_test[m]):.4f}  "
        f"Acc={(preds_test[m] == y_win_test[m]).mean():.4f}  n={m.sum():,}"
    )

## 11 · Plots

In [ ]:
%matplotlib inline
# ── A: LOTO AUC per trial + Optuna history ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4), facecolor=DARK)

ax = axes[0]
ax.set_facecolor(CARD)
ax.bar(TRAIN_TRIALS, loto_aucs, color=TEAL, edgecolor=EDGE, alpha=0.9)
ax.axhline(
    np.mean(loto_aucs),
    color=GOLD,
    lw=2,
    ls="--",
    label=f"Mean={np.mean(loto_aucs):.4f}",
)
ax.set_xticks(TRAIN_TRIALS)
ax.set_xticklabels([f"T{t}" for t in TRAIN_TRIALS], color=WHITE)
ax.set_ylabel("AUC", color=WHITE)
ax.set_xlabel("Held-out trial", color=WHITE)
ax.set_title("LOTO CV — AUC per held-out trial", color=WHITE, fontweight="bold")
ax.tick_params(colors=WHITE)
ax.yaxis.grid(True, color=EDGE)
ax.legend(facecolor=CARD, labelcolor=WHITE, edgecolor=EDGE)
for sp in ax.spines.values():
    sp.set_color(EDGE)

ax = axes[1]
ax.set_facecolor(CARD)
comp = (
    study.trials_dataframe(attrs=("number", "value", "state"))
    .query("state == 'COMPLETE'")
    .sort_values("number")
)
vals, nums = comp["value"].values, comp["number"].values
best_sf = np.maximum.accumulate(vals)
ax.scatter(nums, vals, color=TEAL, alpha=0.45, s=25, zorder=3, label="Trial AUC")
ax.plot(nums, best_sf, color=GOLD, lw=2.5, zorder=4, label="Best so far")
ax.axhline(
    best_sf[-1], color=CORAL, lw=1, ls="--", label=f"Best={best_sf[-1]:.4f}", alpha=0.8
)
ax.set_xlabel("Trial", color=WHITE)
ax.set_ylabel("Validation AUC", color=WHITE)
ax.set_title("Optuna Search History", color=WHITE, fontweight="bold")
ax.tick_params(colors=WHITE)
ax.yaxis.grid(True, color=EDGE)
ax.legend(facecolor=CARD, labelcolor=WHITE, edgecolor=EDGE)
for sp in ax.spines.values():
    sp.set_color(EDGE)

plt.suptitle("Multi-Trial Causal Conv1D", color=WHITE, fontsize=13, fontweight="bold")
plt.tight_layout()

In [ ]:
# ── B: Per-fold LOTO training curves ─────────────────────────────────────
fold_colors = [TEAL, GOLD, LIME, CORAL, "#aa88ff", "#ff88aa", "#88aaff", "#aaffaa"]
fig, ax = plt.subplots(figsize=(14, 5), facecolor=DARK)
ax.set_facecolor(CARD)
for sp in ax.spines.values():
    sp.set_color(EDGE)
for fi, (hist, t) in enumerate(zip(train_hists, TRAIN_TRIALS)):
    ax.plot(
        hist,
        color=fold_colors[fi],
        lw=1.8,
        alpha=0.85,
        label=f"held-out T{t}  (best={max(hist):.4f})",
    )
ax.set_xlabel("Epoch", color=WHITE, fontsize=11)
ax.set_ylabel("Validation AUC", color=WHITE, fontsize=11)
ax.set_title(
    "LOTO Training Curves — Conv1D", color=WHITE, fontsize=13, fontweight="bold"
)
ax.tick_params(colors=WHITE)
ax.yaxis.grid(True, color=EDGE)
ax.legend(facecolor=CARD, labelcolor=WHITE, edgecolor=EDGE, fontsize=8, ncol=2)
plt.tight_layout()

In [ ]:
# ── C: Test trial probability + label timeline ────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(18, 7), sharex=True, facecolor=DARK)
t_ax = np.arange(len(probs_test))
split = int((g_win_test == TEST_TRIALS[0]).sum())

ax = axes[0]
ax.set_facecolor(CARD)
ax.plot(t_ax, probs_test, color=TEAL, lw=0.6, label="P(open)")
ax.axhline(
    best_thresh, color=CORAL, lw=1.2, ls="--", label=f"Threshold={best_thresh:.3f}"
)
ax.axvline(split, color=GOLD, lw=1.5, ls=":", label=f"Trial {TEST_TRIALS[1]} start")
ax.set_ylim(0, 1)
ax.set_ylabel("P(open)", color=WHITE, fontsize=9)
ax.tick_params(colors=WHITE)
ax.legend(facecolor=CARD, labelcolor=WHITE, edgecolor=EDGE, fontsize=8)
for sp in ax.spines.values():
    sp.set_color(EDGE)

ax = axes[1]
ax.set_facecolor(CARD)
ax.plot(
    t_ax, preds_test, drawstyle="steps-post", color=CORAL, lw=1.5, label="Predicted"
)
ax.plot(
    t_ax,
    y_win_test,
    drawstyle="steps-post",
    color=WHITE,
    lw=1.0,
    alpha=0.5,
    label="True",
)
ax.axvline(split, color=GOLD, lw=1.5, ls=":")
ax.set_ylim(-0.1, 1.1)
ax.set_yticks([0, 1])
ax.set_ylabel("Label", color=WHITE, fontsize=9)
ax.tick_params(colors=WHITE)
ax.legend(facecolor=CARD, labelcolor=WHITE, edgecolor=EDGE, fontsize=8)
for sp in ax.spines.values():
    sp.set_color(EDGE)

ax = axes[2]
ax.set_facecolor(CARD)
errors = (preds_test != y_win_test).astype(float)
ax.fill_between(t_ax, errors, color=GOLD, alpha=0.7, step="post", label="Error")
ax.axvline(split, color=GOLD, lw=1.5, ls=":")
ax.set_ylim(0, 1.2)
ax.set_yticks([])
ax.set_ylabel("Error", color=WHITE, fontsize=9)
ax.set_xlabel("Sample (trials 8 & 9 concatenated)", color=WHITE, fontsize=9)
ax.tick_params(colors=WHITE)
ax.legend(facecolor=CARD, labelcolor=WHITE, edgecolor=EDGE, fontsize=8)
for sp in ax.spines.values():
    sp.set_color(EDGE)

fig.suptitle(
    f"Held-Out Test (trials 8 & 9) · AUC={auc_test:.4f}  Acc={acc_test:.4f}",
    color=WHITE,
    fontsize=13,
    fontweight="bold",
)
plt.tight_layout()

In [ ]:
# ── D: Confusion matrices — LOTO vs Test ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4), facecolor=DARK)
for ax, y_true, y_pred, title in [
    (axes[0], y_win[mask], preds_opt, f"LOTO CV  (trials {TRAIN_TRIALS})"),
    (axes[1], y_win_test, preds_test, f"Held-out test  (trials {TEST_TRIALS})"),
]:
    ax.set_facecolor(CARD)
    for sp in ax.spines.values():
        sp.set_color(EDGE)
    ConfusionMatrixDisplay(
        confusion_matrix(y_true, y_pred), display_labels=["Closed", "Open"]
    ).plot(ax=ax, colorbar=False, cmap="YlOrRd")
    ax.set_title(title, color=WHITE, fontsize=10, fontweight="bold")
    ax.tick_params(colors=WHITE)
    ax.xaxis.label.set_color(WHITE)
    ax.yaxis.label.set_color(WHITE)
    for txt in ax.texts:
        txt.set_color("black")
        txt.set_fontsize(12)
plt.suptitle("Confusion Matrices — Conv1D", color=WHITE, fontsize=13, fontweight="bold")
plt.tight_layout()

## 12 · Results Summary & Save

In [ ]:
print("=" * 60)
print("  EEG Causal Conv1D — Multi-Trial Results")
print("=" * 60)
print(f"  Train trials  : {TRAIN_TRIALS}")
print(f"  Test  trials  : {TEST_TRIALS}")
print(f"  Look-back T   : {T} samples  ({T/FS*1000:.0f} ms)")
print(f"  Input shape   : (batch, {N_CHANNELS}, {T})")
print()
print(f"  Optuna best AUC (inner CV) : {study.best_value:.4f}")
print(
    f"  LOTO  AUC (outer CV)       : {np.mean(loto_aucs):.4f} ± {np.std(loto_aucs):.4f}"
)
print(f"  Optimal threshold          : {best_thresh:.3f}")
print(f"  LOTO  Accuracy @ thresh    : {acc_opt:.4f}")
print()
print(f"  TEST  AUC                  : {auc_test:.4f}")
print(f"  TEST  Accuracy @ thresh    : {acc_test:.4f}")
print()
print("  Architecture")
print(f"    Conv layers  : {n_layers} × {n_filters} filters  kernel={kernel_size}")
print(f"    Grow filters : {grow_filters}  BN: {use_bn}  Dropout: {dropout:.3f}")
print(f"    Activation   : {activation}")
print(f"    Optimizer    : {opt_name}  lr={lr:.1e}  wd={wd:.1e}  batch={batch_size}")
print("=" * 60)

In [ ]:
import json

SAVE_PATH = f"../data/subject{SUBJECT}/model_conv1d"
os.makedirs(SAVE_PATH, exist_ok=True)

# Save model weights
torch.save(final_model.state_dict(), f"{SAVE_PATH}/eeg_conv1d_model.pt")

# Save arch + inference config as JSON
params_to_save = {
    "T": int(T),
    "n_channels": int(N_CHANNELS),
    "channel_names": EEG_COLS,
    "fs": int(FS),
    "n_layers": int(n_layers),
    "n_filters": int(n_filters),
    "kernel_size": int(kernel_size),
    "grow_filters": bool(grow_filters),
    "dropout": float(dropout),
    "activation": activation,
    "use_bn": bool(use_bn),
    "best_thresh": float(best_thresh),
    "loto_auc_mean": float(np.mean(loto_aucs)),
    "loto_auc_std": float(np.std(loto_aucs)),
    "test_auc": float(auc_test),
    "train_trials": TRAIN_TRIALS,
    "test_trials": TEST_TRIALS,
    "per_window_norm": True,
}
with open(f"{SAVE_PATH}/eeg_conv1d_params.json", "w") as f:
    json.dump(params_to_save, f, indent=2)

print("Saved:")
print(f"  {SAVE_PATH}/eeg_conv1d_model.pt")
print(f"  {SAVE_PATH}/eeg_conv1d_params.json")